In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Concatenate
import pickle
import os

class HelpdeskPredictor:
    def __init__(self, max_seq_length=10, embedding_dim=50):
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        self.models = {}  # Dictionary to store models for each bucket
        self.activity_encoder = LabelEncoder()
        self.feature_encoders = {}
        
    def prepare_data(self, file_path):
        """Load and prepare the data"""
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")
            
        df = pd.read_csv(file_path)
        df['Complete Timestamp'] = pd.to_datetime(df['Complete Timestamp'])
        df = df.sort_values(['Case ID', 'Complete Timestamp'])

        # Activity encoding
        df['Activity_encoded'] = self.activity_encoder.fit_transform(df['Activity']).astype(np.int32)
        
        # Feature encoding
        categorical_features = ['seriousness', 'service_level', 'service_type', 'workgroup']
        for feature in categorical_features:
            encoder = LabelEncoder()
            df[f'{feature}_encoded'] = encoder.fit_transform(df[feature]).astype(np.int32)
            self.feature_encoders[feature] = encoder
            
        print(f"\nDataset Statistics:")
        print(f"Total events: {len(df)}")
        print(f"Unique cases: {df['Case ID'].nunique()}")
        print(f"Unique activities: {df['Activity'].nunique()}")
        print("\nActivity distribution:")
        print(df['Activity'].value_counts())
            
        return df

    def create_sequences(self, df):
        """Create sequences for training"""
        sequences = []
        next_activities = []
        features = []
        case_ids = []
        
        encoded_features = ['seriousness_encoded', 'service_level_encoded', 
                          'service_type_encoded', 'workgroup_encoded']

        for case_id in df['Case ID'].unique():
            case_df = df[df['Case ID'] == case_id]
            activities = case_df['Activity_encoded'].values.astype(np.int32)
            
            for i in range(1, len(activities)):
                seq = activities[max(0, i-self.max_seq_length):i]
                sequences.append(seq)
                next_activities.append(activities[i])
                feat = case_df.iloc[i-1][encoded_features].values.astype(np.float32)
                features.append(feat)
                case_ids.append(case_id)

        X_seq = pad_sequences(sequences, maxlen=self.max_seq_length, 
                            padding='pre', dtype='int32')
        X_feat = np.array(features, dtype='float32')
        y = np.array(next_activities, dtype='int32')
        
        return X_seq, X_feat, y, case_ids

    def create_prefix_length_buckets(self, X_seq, X_feat, y):
        """Group sequences into buckets based on their length"""
        buckets = {}
        for i in range(len(X_seq)):
            # Calculate actual sequence length (excluding padding)
            seq_length = sum(1 for x in X_seq[i] if x != 0)
            
            if seq_length not in buckets:
                buckets[seq_length] = {
                    'X_seq': [],
                    'X_feat': [],
                    'y': []
                }
            
            buckets[seq_length]['X_seq'].append(X_seq[i])
            buckets[seq_length]['X_feat'].append(X_feat[i])
            buckets[seq_length]['y'].append(y[i])
            
        # Convert lists to numpy arrays
        for length in buckets:
            buckets[length]['X_seq'] = np.array(buckets[length]['X_seq'])
            buckets[length]['X_feat'] = np.array(buckets[length]['X_feat'])
            buckets[length]['y'] = np.array(buckets[length]['y'])
            
        return buckets

    def build_model(self, n_activities, n_features):
        """Build the model architecture"""
        # Sequence input
        seq_input = Input(shape=(self.max_seq_length,), dtype='int32', name='sequence_input')
        x = Embedding(input_dim=n_activities, output_dim=self.embedding_dim)(seq_input)
        x = LSTM(100)(x)
        
        # Feature input
        feat_input = Input(shape=(n_features,), dtype='float32', name='feature_input')
        x = Concatenate()([x, feat_input])
        
        # Dense layers
        x = Dense(100, activation='relu')(x)
        x = Dropout(0.2)(x)
        x = Dense(50, activation='relu')(x)
        x = Dropout(0.2)(x)
        
        # Output layer
        output = Dense(n_activities, activation='softmax')(x)
        
        # Create model
        model = Model(inputs=[seq_input, feat_input], outputs=output)
        model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])
        
        return model

    def evaluate_bucket(self, length, bucket_data):
        """Evaluate model performance for a specific bucket"""
        y_pred = self.models[length].predict([bucket_data['X_seq'], bucket_data['X_feat']])
        y_pred_classes = np.argmax(y_pred, axis=1)
        
        # Convert encoded activities back to names
        activity_names = self.activity_encoder.classes_
        y_test_names = [activity_names[i] for i in bucket_data['y']]
        y_pred_names = [activity_names[i] for i in y_pred_classes]
        
        # Calculate metrics
        metrics = {
            'bucket_length': length,
            'size': len(bucket_data['y']),
            'accuracy': accuracy_score(y_test_names, y_pred_names),
            'f1': f1_score(y_test_names, y_pred_names, average='weighted')
        }
        
        return metrics

    def aggregate_metrics(self, bucket_metrics):
        """Aggregate metrics across all buckets"""
        df_metrics = pd.DataFrame(bucket_metrics)
        
        # Calculate weighted averages
        total_size = df_metrics['size'].sum()
        weighted_accuracy = (df_metrics['accuracy'] * df_metrics['size']).sum() / total_size
        weighted_f1 = (df_metrics['f1'] * df_metrics['size']).sum() / total_size
        
        return {
            'weighted_accuracy': weighted_accuracy,
            'weighted_f1': weighted_f1,
            'bucket_metrics': df_metrics
        }

    def train_and_evaluate(self, X_seq, X_feat, y, epochs=50, batch_size=32, validation_split=0.2):
        """Train and evaluate models for each bucket"""
        # Create buckets
        buckets = self.create_prefix_length_buckets(X_seq, X_feat, y)
        print(f"\nCreated {len(buckets)} prefix length buckets")
        
        overall_metrics = []
        
        # Train a separate model for each bucket
        for length, bucket_data in buckets.items():
            print(f"\nTraining model for prefix length {length}")
            print(f"Bucket size: {len(bucket_data['X_seq'])} sequences")
            
            if len(bucket_data['X_seq']) < batch_size:
                print(f"Skipping bucket {length} - insufficient data")
                continue
                
            # Build model for this bucket
            n_activities = len(self.activity_encoder.classes_)
            n_features = bucket_data['X_feat'].shape[1]
            self.models[length] = self.build_model(n_activities, n_features)
            
            # Train the model
            history = self.models[length].fit(
                [bucket_data['X_seq'], bucket_data['X_feat']],
                bucket_data['y'],
                epochs=epochs,
                batch_size=batch_size,
                validation_split=validation_split,
                verbose=1
            )
            
            # Evaluate the model
            metrics = self.evaluate_bucket(length, bucket_data)
            overall_metrics.append(metrics)
            
        # Aggregate metrics across all buckets
        return self.aggregate_metrics(overall_metrics)

    def predict_next(self, sequence, features):
        """Predict next activity with probability using appropriate bucket"""
        if not isinstance(sequence, list):
            sequence = [sequence]
            
        # Calculate sequence length to select appropriate model
        seq_length = len(sequence)
        
        if seq_length not in self.models:
            # If no exact match, use the closest available model
            available_lengths = sorted(self.models.keys())
            seq_length = min(available_lengths, key=lambda x: abs(x - seq_length))
            
        # Convert sequence to encoded form
        seq_encoded = self.activity_encoder.transform(sequence)
        X_seq = pad_sequences([seq_encoded], maxlen=self.max_seq_length, 
                            padding='pre', dtype='int32')
        X_feat = np.array([features], dtype='float32')
        
        # Get predictions using the appropriate model
        pred_probs = self.models[seq_length].predict([X_seq, X_feat])[0]
        
        # Get top 3 predictions
        top_indices = pred_probs.argsort()[-3:][::-1]
        predictions = []
        for idx in top_indices:
            activity = self.activity_encoder.inverse_transform([idx])[0]
            probability = pred_probs[idx]
            predictions.append((activity, probability))
        
        return predictions

    def save_model(self, directory):
        """Save all models and parameters"""
        if not os.path.exists(directory):
            os.makedirs(directory)
            
        # Save each bucket model
        for length, model in self.models.items():
            model_path = os.path.join(directory, f'helpdesk_model_bucket_{length}.keras')
            model.save(model_path)
        
        # Save state
        state = {
            'activity_encoder': self.activity_encoder,
            'feature_encoders': self.feature_encoders,
            'max_seq_length': self.max_seq_length,
            'embedding_dim': self.embedding_dim,
            'bucket_lengths': list(self.models.keys())
        }
        state_path = os.path.join(directory, 'model_state.pkl')
        with open(state_path, 'wb') as f:
            pickle.dump(state, f)
            
        print(f"\nModels and state saved in: {directory}")

    @classmethod
    def load_model(cls, directory):
        """Load all saved models and parameters"""
        with open(os.path.join(directory, 'model_state.pkl'), 'rb') as f:
            state = pickle.load(f)
            
        instance = cls(
            max_seq_length=state['max_seq_length'],
            embedding_dim=state['embedding_dim']
        )
        
        instance.activity_encoder = state['activity_encoder']
        instance.feature_encoders = state['feature_encoders']
        
        # Load each bucket model
        for length in state['bucket_lengths']:
            model_path = os.path.join(directory, f'helpdesk_model_bucket_{length}.keras')
            instance.models[length] = load_model(model_path)
        
        return instance

def analyze_real_examples(predictor, df):
    """Analyze real examples from the dataset"""
    print("\nReal Examples Analysis:")
    print("======================")
    
    # Get random cases
    random_cases = np.random.choice(df['Case ID'].unique(), 3, replace=False)
    
    for case_id in random_cases:
        case_df = df[df['Case ID'] == case_id].copy()
        case_df = case_df.sort_values('Complete Timestamp')
        
        print(f"\nCase ID: {case_id}")
        activities = case_df['Activity'].values
        
        # Analyze sequence points
        for i in range(2, len(activities)):
            current_sequence = activities[max(0, i-3):i].tolist()  # Convert to list
            actual_next = activities[i]
            
            # Use actual features from the dataset
            feature_values = case_df.iloc[i-1][[f'{feat}_encoded' for feat in 
                ['seriousness', 'service_level', 'service_type', 'workgroup']]].values
            
            # Get predictions
            try:
                predictions = predictor.predict_next(current_sequence, feature_values)
                
                print(f"\nCurrent sequence: {' -> '.join(current_sequence)}")
                print(f"Actual next activity: {actual_next}")
                print("Top 3 predictions (with probabilities):")
                for act, prob in predictions:
                    correct_mark = "✓" if act == actual_next else " "
                    print(f"{correct_mark} {act}: {prob:.2f}")
                print("-" * 50)
            except Exception as e:
                print(f"Error during prediction: {str(e)}")
                continue

def test_specific_sequences(predictor):
    """Test model with specific sequences"""
    print("\nTesting Specific Sequences:")
    print("==========================")
    
    test_cases = [
        (['Insert ticket', 'Assign seriousness'], "New ticket flow"),
        (['Take in charge ticket', 'Wait'], "Waiting flow"),
        (['Create SW anomaly', 'Wait', 'Resolve ticket'], "Software anomaly flow")
    ]
    
    # Use median values for features as default
    default_features = np.array([1, 1, 1, 1], dtype=np.float32)
    
    for sequence, desc in test_cases:
        print(f"\nTesting {desc}:")
        print(f"Sequence: {' -> '.join(sequence)}")
        
        try:
            predictions = predictor.predict_next(sequence, default_features)
            print("Predictions:")
            for activity, prob in predictions:
                print(f"- {activity}: {prob:.2f}")
            print("-" * 50)
        except Exception as e:
            print(f"Error during prediction: {str(e)}")
            continue

def main():
   """Main execution function"""
   try:
       # Set up file paths
       current_dir = os.getcwd()
       data_path = os.path.join(current_dir, 'dataset', 'HelpDesk', 'finale.csv')
       model_save_dir = os.path.join(current_dir, 'models')
       
       # Initialize predictor
       predictor = HelpdeskPredictor()
       
       # Check if data file exists
       if not os.path.exists(data_path):
           raise FileNotFoundError(f"Data file not found at: {data_path}")
       
       print(f"Loading data from {data_path}...")
       df = predictor.prepare_data(data_path)
       
       print("\nPreparing sequences...")
       X_seq, X_feat, y, case_ids = predictor.create_sequences(df)
       
       print(f"\nTraining Data Shape:")
       print(f"X_seq shape: {X_seq.shape}")
       print(f"X_feat shape: {X_feat.shape}")
       print(f"y shape: {y.shape}")
       
       print("\nTraining models...")
       metrics = predictor.train_and_evaluate(X_seq, X_feat, y, epochs=50)
       
       print("\nBucket-wise Performance:")
       print(metrics['bucket_metrics'])
       print(f"\nOverall weighted accuracy: {metrics['weighted_accuracy']:.3f}")
       print(f"Overall weighted F1-score: {metrics['weighted_f1']:.3f}")
       
       # Create models directory if it doesn't exist
       if not os.path.exists(model_save_dir):
           os.makedirs(model_save_dir)
       
       # Save models
       predictor.save_model(model_save_dir)
       
       # Save metrics to CSV
       metrics_path = os.path.join(model_save_dir, 'activity_metrics.csv')
       metrics['bucket_metrics'].to_csv(metrics_path, index=False)
       print(f"\nMetrics saved to: {metrics_path}")
       
       # Analyze examples
       analyze_real_examples(predictor, df)
       
       # Test specific sequences
       test_specific_sequences(predictor)
       
   except Exception as e:
       print(f"Error: {str(e)}")
       import traceback
       traceback.print_exc()

if __name__ == "__main__":
   main()

Loading data from /Users/ivan/Desktop/MITLxPPM/MITLxPPM/dataset/HelpDesk/finale.csv...

Dataset Statistics:
Total events: 21348
Unique cases: 4580
Unique activities: 14

Activity distribution:
Activity
Take in charge ticket    5060
Resolve ticket           4983
Assign seriousness       4938
Closed                   4574
Wait                     1463
Require upgrade           119
Insert ticket             118
Create SW anomaly          67
Resolve SW anomaly         13
Schedule intervention       5
VERIFIED                    3
RESOLVED                    2
INVALID                     2
DUPLICATE                   1
Name: count, dtype: int64

Preparing sequences...

Training Data Shape:
X_seq shape: (16768, 10)
X_feat shape: (16768, 4)
y shape: (16768,)

Training models...

Created 11 prefix length buckets

Training model for prefix length 0
Bucket size: 4797 sequences
Epoch 1/50
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7040 - loss: 1.2900 - val_accuracy: 0.8500 - val_loss: